# Part b): Writing our own neural network code

We fit the one-dimensional Runge function with our own feed-forward neural network (FFNN) and compare it with the best OLS model from project 1. The network classes live in `fys_stk4155_p2.nn`, while the data generation, optimizers and OLS code are imported from project 1 (`fys_stk4155_p1`, release v0.1.0).

In [ ]:
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scienceplots  # noqa: F401
from fys_stk4155_p1.data.design_matrix import univariate_polynomial_design_matrix
from fys_stk4155_p1.data.runge import generate_runge_data, runge_function
from fys_stk4155_p1.metrics import mean_squared_error, r2_score
from fys_stk4155_p1.optimization.optimizers import Adam, Optimizer, Plain, RMSProp
from fys_stk4155_p1.regression.degree_sweep import fit_polynomial_degree_sweep
from fys_stk4155_p1.regression.ordinary_least_squares import OLS
from joblib import Parallel, delayed
from matplotlib.colors import LogNorm
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler

from fys_stk4155_p2.nn.activations import Activation, LeakyReLU, ReLU, Sigmoid
from fys_stk4155_p2.nn.layers import Dense, Layer, Linear
from fys_stk4155_p2.nn.losses import MeanSquaredError
from fys_stk4155_p2.nn.network import FFNN

plt.style.use("ieee")

## Configuration

The data settings are those of the reference data set in project 1, so that the networks and OLS are compared on the same data and the same train-test split.

In [ ]:
# Data, identical to the reference data set of project 1
N_SAMPLES = 100
NOISE_STD = 0.1
SEED = 42
TEST_SIZE = 0.2

# Polynomial degree with the lowest test MSE on the reference split in project 1
OLS_DEGREE = 12

# Hyperparameter grid for the networks, searched with k-fold cross-validation
EPOCHS = [100, 1000, 10000]
BATCH_SIZE = [None, 8, 16, 32]  # None: full-batch gradient descent
LEARNING_RATE = [0.1, 0.01, 0.001, 0.0001]
OPTIMIZER = [Plain, RMSProp, Adam]
N_FOLDS = 5

# Cached grid-search results; recomputed automatically if the grid above changes,
# but not if the model code changes: delete the file to rerun the search then
GRID_SEARCH_PATH = Path("../data/results/b_grid_search.csv")

## Data

We sample the Runge function with Gaussian noise and split the samples into training and test data.

The network takes the raw $x$ as its only input feature, so the design matrix is the single column $x$ rather than the polynomial expansion used in project 1: the hidden layers learn the nonlinear features themselves. We standardize $x$ with the mean and standard deviation of the training data only, so that no information from the test data enters the training.

In [ ]:
x, y = generate_runge_data(n=N_SAMPLES, noise_std=NOISE_STD, seed=SEED)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=TEST_SIZE, random_state=SEED)

scaler = StandardScaler().fit(x_train.reshape(-1, 1))
X_train = scaler.transform(x_train.reshape(-1, 1))
X_test = scaler.transform(x_test.reshape(-1, 1))

## OLS baseline from project 1

We refit OLS with the polynomial degree selected in project 1. `fit_polynomial_degree_sweep` builds the polynomial design matrix, splits it with the same seed and test size as above (giving the same training and test samples), and standardizes the features on the training data.

In [ ]:
ols_results = fit_polynomial_degree_sweep(x, y, [OLS_DEGREE], OLS, test_size=TEST_SIZE, seed=SEED)
print(
    f"OLS (p = {OLS_DEGREE}): test MSE = {ols_results['mse_test'][0]:.5f}, "
    f"test R2 = {ols_results['r2_test'][0]:.4f}"
)

## Feed-forward neural networks

`create_ffnn` builds a network with sigmoid activations in the hidden layers and a linear output layer, since the target is an unbounded real number. The weights are drawn from a normal distribution whose width depends on the number of inputs to the layer (Section 8.10 of the lecture notes). For the sigmoid we use Xavier initialization, $\sigma_W^2 = 2/(n_\text{in} + n_\text{out})$, which keeps the pre-activations near zero, where the sigmoid is close to linear and its derivative is largest; the He initialization $\sigma_W^2 = 2/n_\text{in}$ is meant for the ReLU family (part c), which zeroes about half of its inputs. The biases are initialized to zero, as the random weights already break the symmetry between the neurons.

In [ ]:
# Weight initialization of the hidden layers, matched to their activation (Section 8.10)
INITIALIZATION = {Sigmoid: "xavier", ReLU: "he", LeakyReLU: "he"}


def create_ffnn(
    hidden_layers: list[int],
    optimizer: Optimizer,
    activation: type[Activation] = Sigmoid,
    *,
    rng: np.random.Generator,
) -> FFNN:
    """Build an FFNN for the one-dimensional Runge function, trained with the MSE.

    Args:
        hidden_layers: Number of neurons in each hidden layer.
        optimizer: Fresh optimizer instance, copied for every weight matrix and bias.
        activation: The activation function class which to use for the hidden layers.
        rng: Random number generator for the initial weights.

    Returns:
        The untrained network.
    """
    layers: list[Layer] = []
    n_inputs = 1  # x is the only input feature
    for n_neurons in hidden_layers:
        layers += [Dense(n_inputs, n_neurons, INITIALIZATION[activation], rng=rng), activation()]
        n_inputs = n_neurons
    layers += [Dense(n_inputs, 1, rng=rng), Linear()]

    return FFNN(layers, MeanSquaredError(), optimizer)

## Hyperparameter search

We search over the number of epochs, the batch size, the learning rate and the optimizer for both architectures: plain gradient descent, and RMSprop and Adam. A batch size of `None` means full-batch training, so `Plain` with `None` is plain gradient descent, and the other batch sizes give stochastic gradient descent.

Each configuration is scored by its mean validation MSE over 5-fold cross-validation on the training data, as for the model selection in project 1, so that the test data is used only for the final comparison. Within each fold, $x$ is standardized with the statistics of that fold's training part. All configurations start from the same initial weights, so that differences come from the hyperparameters alone. A configuration whose predictions overflow is marked as diverged, with an infinite MSE.

The search trains about 2900 networks, so it runs in parallel on all cores and caches its results in `GRID_SEARCH_PATH`.

In [ ]:
architectures = {
    "1 hidden layer (50 nodes)": [50],
    "2 hidden layers (100 nodes)": [100, 100],
}
OPTIMIZERS_BY_NAME = {optimizer.__name__: optimizer for optimizer in OPTIMIZER}
GRID_COLUMNS = ["architecture", "optimizer", "learning_rate", "batch_size", "epochs"]


def as_batch_size(value: float) -> int | None:
    """Convert a batch size from the results table, where full-batch is missing, to fit's form."""
    return None if pd.isna(value) else int(value)


def cv_mse(
    architecture: str, optimizer: str, learning_rate: float, batch_size: int | None, epochs: int
) -> float:
    """Mean validation MSE over the k folds of the training data, or inf if training diverges."""
    fold_mse = []
    for fit_idx, val_idx in KFold(N_FOLDS, shuffle=True, random_state=SEED).split(x_train):
        fold_scaler = StandardScaler().fit(x_train[fit_idx].reshape(-1, 1))
        network = create_ffnn(
            hidden_layers=architectures[architecture],
            optimizer=OPTIMIZERS_BY_NAME[optimizer](learning_rate=learning_rate),
            rng=np.random.default_rng(SEED),
        )
        with np.errstate(over="ignore", invalid="ignore"):
            network.fit(
                fold_scaler.transform(x_train[fit_idx].reshape(-1, 1)),
                y_train[fit_idx],
                epochs=epochs,
                batch_size=batch_size,
            )
            prediction = network.predict(fold_scaler.transform(x_train[val_idx].reshape(-1, 1)))
        if not np.all(np.isfinite(prediction)):
            return np.inf
        fold_mse.append(mean_squared_error(y_train[val_idx], prediction))
    return float(np.mean(fold_mse))


grid = pd.DataFrame(
    product(architectures, OPTIMIZERS_BY_NAME, LEARNING_RATE, BATCH_SIZE, EPOCHS),
    columns=GRID_COLUMNS,
).astype({"batch_size": "Int64"})

cached = (
    pd.read_csv(GRID_SEARCH_PATH, dtype={"batch_size": "Int64"})
    if GRID_SEARCH_PATH.exists()
    else None
)
if cached is not None and cached[GRID_COLUMNS].equals(grid):
    grid_results = cached
else:
    scores = Parallel(n_jobs=-1, verbose=1)(
        delayed(cv_mse)(
            row.architecture,
            row.optimizer,
            row.learning_rate,
            as_batch_size(row.batch_size),
            int(row.epochs),
        )
        for row in grid.itertuples(index=False)
    )
    grid_results = grid.assign(cv_mse=scores)
    GRID_SEARCH_PATH.parent.mkdir(parents=True, exist_ok=True)
    grid_results.to_csv(GRID_SEARCH_PATH, index=False)

The best configuration for each combination of architecture and optimizer, sorted by the cross-validated MSE:

In [ ]:
best_configs = grid_results.loc[
    grid_results.groupby(["architecture", "optimizer"])["cv_mse"].idxmin()
].sort_values("cv_mse")
best_configs.round({"cv_mse": 5})

## Learning rates and optimizers

The cross-validated MSE as a function of the learning rate $\gamma$, for each optimizer with full-batch training (gradient descent, GD) and with minibatches (stochastic gradient descent, SGD). Each cell shows the best MSE over the number of epochs and, for SGD, the batch size. A cell is marked "div." if all its configurations diverged.

In [ ]:
METHOD_ORDER = [f"{opt}, {method}" for opt in OPTIMIZERS_BY_NAME for method in ("GD", "SGD")]
learning_rates = sorted(LEARNING_RATE)

lr_table = grid_results.assign(
    method=grid_results["optimizer"] + np.where(grid_results["batch_size"].isna(), ", GD", ", SGD")
).pivot_table(
    index=["architecture", "method"], columns="learning_rate", values="cv_mse", aggfunc="min"
)

finite = lr_table.to_numpy()[np.isfinite(lr_table.to_numpy())]
norm = LogNorm(vmin=finite.min(), vmax=finite.max())
cmap = plt.get_cmap("Blues").with_extremes(bad="#d3d1c7")

fig, axes = plt.subplots(
    1, len(architectures), figsize=(7, 3.4), sharey=True, constrained_layout=True
)
for ax, name in zip(axes, architectures, strict=True):
    table = lr_table.loc[name].loc[METHOD_ORDER, learning_rates].to_numpy()
    image = ax.imshow(np.ma.masked_invalid(table), cmap=cmap, norm=norm, aspect="auto")
    for (i, j), value in np.ndenumerate(table):
        if np.isfinite(value):
            text, color = f"{value:.4f}", "white" if norm(value) > 0.6 else "#1f1e1d"
        else:
            text, color = "div.", "#1f1e1d"
        ax.text(j, i, text, ha="center", va="center", fontsize=7, color=color)
    ax.set_xticks(
        range(len(learning_rates)), [f"$10^{{{round(np.log10(lr))}}}$" for lr in learning_rates]
    )
    ax.set_yticks(range(len(METHOD_ORDER)), METHOD_ORDER)
    ax.set_xlabel(r"Learning rate $\gamma$")
    ax.set_title(name, fontsize=9)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
fig.colorbar(image, ax=axes, label="Cross-validated MSE", shrink=0.9)
fig.savefig("../docs/figures/regression/learning_rates.pdf")
plt.show()

The heat map shows only the end point of training. To see how each optimizer gets there, we train the network with one hidden layer on all the training data with SGD (batch size 16) for every learning rate in the grid and record the training MSE after each epoch. Curves that stop have diverged.

In [ ]:
CURVE_ARCHITECTURE = "1 hidden layer (50 nodes)"
CURVE_BATCH_SIZE = 16
CURVE_EPOCHS = max(EPOCHS)


def training_curve(optimizer: str, learning_rate: float) -> list[float]:
    """Training MSE per epoch for the curve architecture, trained on all training data."""
    network = create_ffnn(
        hidden_layers=architectures[CURVE_ARCHITECTURE],
        optimizer=OPTIMIZERS_BY_NAME[optimizer](learning_rate=learning_rate),
        rng=np.random.default_rng(SEED),
    )
    with np.errstate(over="ignore", invalid="ignore"):
        return network.fit(X_train, y_train, epochs=CURVE_EPOCHS, batch_size=CURVE_BATCH_SIZE)


curve_configs = list(product(OPTIMIZERS_BY_NAME, learning_rates))
curves = dict(
    zip(
        curve_configs,
        Parallel(n_jobs=-1)(delayed(training_curve)(*config) for config in curve_configs),
        strict=True,
    )
)

# One sequential hue, light to dark with increasing learning rate
lr_shades = plt.get_cmap("Blues")(np.linspace(0.4, 1.0, len(learning_rates)))
lr_colors = dict(zip(learning_rates, lr_shades, strict=True))

fig, axes = plt.subplots(
    1, len(OPTIMIZERS_BY_NAME), figsize=(7, 3), sharey=True, constrained_layout=True
)
for ax, optimizer in zip(axes, OPTIMIZERS_BY_NAME, strict=True):
    for learning_rate in learning_rates:
        history = curves[optimizer, learning_rate]
        ax.plot(
            np.arange(1, len(history) + 1),
            history,
            lw=1.2,
            color=lr_colors[learning_rate],
            label=rf"$\gamma = 10^{{{round(np.log10(learning_rate))}}}$",
        )
    ax.axhline(NOISE_STD**2, color="black", ls=":", lw=0.8, label=r"noise floor $\sigma^2$")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(optimizer, fontsize=9)
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.3, lw=0.5)
axes[0].set_ylabel("Training MSE")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels, loc="outside lower center", ncols=len(labels), frameon=False, fontsize=8
)
fig.savefig("../docs/figures/regression/optimizer_curves.pdf")
plt.show()

## Training the selected networks

For each architecture, we retrain the configuration with the lowest cross-validated MSE on all the training data. These networks are compared with OLS below.

In [ ]:
best_per_architecture = grid_results.loc[
    grid_results.groupby("architecture")["cv_mse"].idxmin()
].set_index("architecture")

networks: dict[str, FFNN] = {}
histories: dict[str, list[float]] = {}

for name, hidden_layers in architectures.items():
    config = best_per_architecture.loc[name]
    print(config)
    network = create_ffnn(
        hidden_layers=hidden_layers,
        optimizer=OPTIMIZERS_BY_NAME[config.optimizer](learning_rate=config.learning_rate),
        rng=np.random.default_rng(SEED),
    )
    histories[name] = network.fit(
        X_train, y_train, epochs=int(config.epochs), batch_size=as_batch_size(config.batch_size)
    )
    networks[name] = network

## Comparison of the fitted models

We evaluate OLS and the two networks on a dense grid over $[-1, 1]$ and plot them on top of the Runge function and the data. OLS is refitted directly rather than taken from `ols_results`, since `fit_polynomial_degree_sweep` does not return the scaler of the polynomial features; with the same degree, training data and scaling, it is the same model. The networks were trained on standardized $x$, so the grid is passed through the same `scaler`.

In [ ]:
poly_scaler = StandardScaler().fit(univariate_polynomial_design_matrix(x_train, OLS_DEGREE))


def ols_design(x_values: np.ndarray) -> np.ndarray:
    """Standardized polynomial design matrix, scaled with the training statistics."""
    return poly_scaler.transform(univariate_polynomial_design_matrix(x_values, OLS_DEGREE))


ols = OLS().fit(ols_design(x_train), y_train)

x_plot = np.linspace(-1, 1, 500)
X_plot = scaler.transform(x_plot.reshape(-1, 1))
predictions = {"OLS": ols.predict(ols_design(x_plot))}
predictions |= {name: network.predict(X_plot) for name, network in networks.items()}

MODEL_LABELS = {"OLS": f"OLS ($p = {OLS_DEGREE}$)"} | {name: f"FFNN, {name}" for name in networks}

fig, ax = plt.subplots(figsize=(6, 4.6), constrained_layout=True)
ax.scatter(x_train, y_train, s=10, color="#b4b2a9", label="training data")
ax.scatter(
    x_test, y_test, s=14, facecolors="none", edgecolors="#5f5e5a", linewidths=0.8, label="test data"
)
ax.plot(x_plot, runge_function(x_plot), color="black", lw=1.5, label="Runge function")
for name, prediction in predictions.items():
    ax.plot(x_plot, prediction, lw=1.5, label=MODEL_LABELS[name])
ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.grid(alpha=0.3, lw=0.5)
fig.legend(loc="outside lower center", ncols=2, frameon=False, fontsize=8)
fig.savefig("../docs/figures/regression/model_comparison.pdf")
plt.show()

## Training history

The training MSE after each epoch, with the noise floor $\sigma^2$ as reference: even the true Runge function has an expected MSE of $\sigma^2$ on noisy data. A training MSE well below $\sigma^2$ means the network has started to fit the noise, while one far above it means it still underfits.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
for name, history in histories.items():
    ax.plot(
        np.arange(1, len(history) + 1),
        history,
        lw=1.5,
        label=MODEL_LABELS[name],
    )
ax.axhline(NOISE_STD**2, color="black", ls=":", lw=0.8, label=r"noise floor $\sigma^2$")
ax.set_xlabel("Epoch")
ax.set_xscale("log")
ax.set_ylabel("Training MSE")
ax.set_yscale("log")
ax.grid(alpha=0.3, lw=0.5)
ax.legend(frameon=False, fontsize=8)
fig.savefig("../docs/figures/regression/history.pdf")
plt.show()

## Metrics

MSE and $R^2$ on the training and test data, and the MSE against the true Runge function $f$ on the dense grid. The test set has only 20 samples, so its MSE is noisy and includes the irreducible error $\sigma^2$; the MSE against $f$ measures how well each model has learned the function itself. Note also that the test $R^2$ cannot approach 1 even for a perfect model: its expected value is at most about $1 - \sigma^2 / \mathrm{Var}(y_\text{test})$.

In [ ]:
# Each model as a function of the raw x, so all metrics use the same inputs
fitted = {"OLS": lambda x_values: ols.predict(ols_design(x_values))}
fitted |= {
    name: lambda x_values, net=network: net.predict(scaler.transform(x_values.reshape(-1, 1)))
    for name, network in networks.items()
}

metrics = pd.DataFrame(
    {
        MODEL_LABELS[name]: {
            "train MSE": mean_squared_error(y_train, predict(x_train)),
            "test MSE": mean_squared_error(y_test, predict(x_test)),
            "train R2": r2_score(y_train, predict(x_train)),
            "test R2": r2_score(y_test, predict(x_test)),
            "MSE vs f": mean_squared_error(runge_function(x_plot), predict(x_plot)),
        }
        for name, predict in fitted.items()
    }
).T
metrics.round(4)

# Part c) Testing different activation functions and depths of the neural network

This part asks us to (1) compare Sigmoid, ReLU and Leaky ReLU in the hidden layers, (2) study the result as a function of the number of hidden layers and nodes, and (3) look for overfitting.

In [ ]:
N_HIDDEN_LAYERS = [1, 2, 4, 8]
N_NODES_PER_LAYER = [10, 20, 40, 80, 160, 320]
ACTIVATION = [Sigmoid, ReLU, LeakyReLU]